# Lab 1D2: Indexing Policy in Python

**Time**: ~10 min  
**Environment**: Jupyter kernel in VS Code  

In this exercise you will inspect pre-created Cosmos DB containers with different indexing policies and measure the RU cost impact of excluding large fields from indexing.

## Prerequisites

- Python 3.10+ with `azure-cosmos`, `azure-identity`, and `python-dotenv` installed: `pip install azure-cosmos azure-identity python-dotenv`
- `COSMOS_ENDPOINT` environment variable set to your Cosmos DB account endpoint

Run each cell in order to complete the steps.

## Step 0: Initialize Connection

Set up the Cosmos client connection.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os

ENDPOINT = os.environ.get("COSMOS_ENDPOINT")
DB_NAME = "WorkshopData"

if not ENDPOINT:
    raise RuntimeError("COSMOS_ENDPOINT environment variable is required.")

print(f"Cosmos Endpoint: {ENDPOINT}")
print(f"Database: {DB_NAME}")

In [ ]:
from azure.cosmos import CosmosClient, PartitionKey
from azure.identity import AzureCliCredential

cred = AzureCliCredential()
client = CosmosClient(url=ENDPOINT, credential=cred)
db = client.get_database_client(DB_NAME)
print(f"Connected to: {ENDPOINT}/{DB_NAME}")

## Step 1: Inspect Container with Default Indexing (Prebuilt)

Read `ItemsDefaultIndex` and print its indexing policy.

> **Note**: Containers are deployed in advance via the workshop Bicep template (`bicep/modules/cosmosdb.bicep`) because Cosmos DB AAD tokens only authorize data-plane operations, not control plane operations like creating containers.

In [ ]:
default_container_name = "ItemsDefaultIndex"

container_default = db.get_container_client(default_container_name)
props = container_default.read()
policy = props["indexingPolicy"]

print(f"Container '{default_container_name}' found")
print(f"  Indexing mode: {policy.get('indexingMode')}")
print(f"  Included paths: {[p['path'] for p in policy.get('includedPaths', [])]}")
print(f"  Excluded paths: {[p['path'] for p in policy.get('excludedPaths', [])]}")

## Step 2: Inspect Container with Custom Indexing

Read `ItemsCustomIndex` and verify both `/largeBlob/?` and `/metadata/*` appear in `excludedPaths`. Replace the placeholder list in the code cell with the real read:

```python
props = container_custom.read()
policy = props["indexingPolicy"]
excluded = [p["path"] for p in policy.get("excludedPaths", [])]
```

**Expected output**: Both excluded paths printed; "Custom indexing policy verified."

In [ ]:
custom_container_name = "ItemsCustomIndex"

container_custom = db.get_container_client(custom_container_name)

props = container_custom.read()
policy = props["indexingPolicy"]
excluded = [p["path"] for p in policy.get("excludedPaths", [])]

print(f"Container '{custom_container_name}' found")
print(f"  Excluded paths: {excluded}")

blob_excluded = any(p.startswith("/largeBlob/") for p in excluded)
meta_excluded = any(p.startswith("/metadata/") for p in excluded)
ok = blob_excluded and meta_excluded
print("Custom indexing policy verified." if ok else "WARNING: expected /largeBlob and /metadata exclusions.")

## Step 3: RU Comparison — `largeBlob` only (`?` exclusion) (Prebuilt)

Write a document containing only a ~10 KB scalar string at `/largeBlob` to both containers. The custom container's `/largeBlob/?` exclusion skips indexing that single value.

In [ ]:
import random
import string
import uuid

def compare_ru(label: str, payload: dict):
    item_default = {"id": f"{label}_{uuid.uuid4().hex[:8]}", "partitionKey": "idx", **payload}
    item_custom = {"id": f"{label}_{uuid.uuid4().hex[:8]}", "partitionKey": "idx", **payload}

    container_default.create_item(body=item_default)
    d_ru = float(container_default.client_connection.last_response_headers["x-ms-request-charge"])

    container_custom.create_item(body=item_custom)
    c_ru = float(container_custom.client_connection.last_response_headers["x-ms-request-charge"])

    savings = d_ru - c_ru
    pct = (savings / d_ru) * 100 if d_ru else 0
    print(f"  Default index ~RU: {d_ru:.2f}")
    print(f"  Custom index   ~RU: {c_ru:.2f}")
    print(f"  RU savings: {savings:.2f} RU ({pct:.1f}%)")
    return d_ru, c_ru

# Scalar value at /largeBlob - excluded on the custom container via '/largeBlob/?'.
# Use random characters instead of a single repeating char so the indexer can't
# trivially compress/short-circuit the value.
large_blob = "".join(random.choices(string.ascii_lowercase, k=10000))
compare_ru("blob_test", {"largeBlob": large_blob})

## Step 4: RU Comparison — `metadata` only (`*` exclusion) (Prebuilt)

Write a document with a nested `metadata` object (~50 fields) to both containers. The custom container's `/metadata/*` exclusion skips the entire subtree.

In [ ]:
metadata = {f"tag{i}": f"value_{i}" for i in range(50)}
compare_ru("meta_test", {"metadata": metadata})

## Step 5: RU Comparison — combined (Prebuilt)

Writes a document with **both** `largeBlob` and `metadata`. The savings from Steps 3 and 4 stack.

In [ ]:
compare_ru("both_test", {"largeBlob": large_blob, "metadata": metadata})

print("\n=== Lab Complete ===")
print("You have completed the indexing policy exercise in Python. You:")
print("- Inspected the default-indexing and custom-indexing containers")
print("- Measured RU cost of excluding a large scalar value ('?' path)")
print("- Measured RU cost of excluding a nested object subtree ('*' path)")
print("- Saw how the two patterns combine on a richer document")
print("\nKey takeaway: Excluding large scalar values and noisy nested structures from indexing can significantly reduce RU costs on write operations.")